# SPI Gorgon

### Read Data

In [6]:
import pandas as pd
import adlfs
from dotenv import load_dotenv

def abfss_to_adlfs_path(abfss_path: str) -> str:
    """
    Convert:
      abfss://container@account.dfs.core.windows.net/some/path/
    to:
      container/some/path
    """
    if not abfss_path.startswith("abfss://"):
        raise ValueError(f"Expected abfss:// path, got: {abfss_path}")

    without_scheme = abfss_path.replace("abfss://", "", 1)
    container, rest = without_scheme.split("@", 1)

    marker = ".dfs.core.windows.net/"
    if marker not in rest:
        raise ValueError(f"Unexpected ADLS Gen2 URL (missing {marker}): {abfss_path}")

    path = rest.split(marker, 1)[1]
    return f"{container}/{path}".rstrip("/")


def read_latest_parquet_from_adls(folder_abfss: str, storage_options: dict, **read_parquet_kwargs) -> pd.DataFrame:
    fs = adlfs.AzureBlobFileSystem(**storage_options)

    folder_fs = abfss_to_adlfs_path(folder_abfss).rstrip("/")

    # list with metadata (so we can get last_modified)
    items = fs.ls(folder_fs, detail=True)
    pq_items = [x for x in items if x.get("name", "").lower().endswith(".parquet")]

    if not pq_items:
        raise FileNotFoundError(f"No Parquet files found under: {folder_abfss}")

    latest = max(pq_items, key=lambda x: x.get("last_modified"))
    latest_path = latest["name"]

    print("Latest parquet:", latest_path)
    print("Last modified:", latest.get("last_modified"))

    # IMPORTANT: file handle -> DO NOT pass storage_options to pandas
    with fs.open(latest_path, "rb") as fp:
        return pd.read_parquet(fp, engine="pyarrow", **read_parquet_kwargs)


def main():
    load_dotenv()

    ACCOUNT_NAME = "chevrondatalake"
    CONTAINER_NAME = "produced"
    DIRECTORY_PATH = "Surface/source_hexagonsmartcloud_app3060720/SmartInstrumentation/Gorgon/View/CVX01SI_GORGP.CVX_MTL"

    # folder path where parquet files exist
    base_path = f"abfss://{CONTAINER_NAME}@{ACCOUNT_NAME}.dfs.core.windows.net/{DIRECTORY_PATH}/"

    storage_options = {
        "account_name": ACCOUNT_NAME,
        "anon": False,   # uses your Azure login / environment auth
    }

    # ✅ read ONLY the latest parquet file from folder
    df = read_latest_parquet_from_adls(base_path, storage_options)

    print("SPI:", df.shape)
    return df


# In notebook:
df = main()
df_spi=df

Latest parquet: produced/Surface/source_hexagonsmartcloud_app3060720/SmartInstrumentation/Gorgon/View/CVX01SI_GORGP.CVX_MTL/part-00000-01ee4a1c-c6ae-471e-89a1-f6018ee899e0.c000.snappy.parquet
Last modified: 2026-03-22 00:19:48+00:00
SPI: (277533, 15)


x=df_spi[df_spi[['TAG_NUMBER','CMPNT_ID','CAT']].duplicated(keep=False)].sort_values(by=['TAG_NUMBER','CMPNT_ID','CAT'])
x.to_csv("spi_duplicate.csv")

import pandas as pd
df_spi=pd.read_excel("CVX_MTL_SPI_Gor.xlsx")

### Columns to keep

In [7]:
columns_to_keep=['CAT', 'CMPNT_ID','CMPNT_NAME', 'TAG_NUMBER', 'DISCIPLINE','STATUS', 'PROJECT_NO']
df_spi=df_spi[columns_to_keep]
rename_map={'CAT':'category', 'CMPNT_ID':'componentId','CMPNT_NAME':'componentName',  'TAG_NUMBER':'tagNumber', 'DISCIPLINE': 'discipline','STATUS':'status', 'PROJECT_NO': 'projectNumber'}
df_spi=df_spi.rename(columns=rename_map)
#df_spi.to_csv('SPI_CVX_MTL.tsv', sep="\t", index=False)

In [8]:
x=df_spi.sort_values("tagNumber")
x[x.duplicated(subset=["tagNumber","componentId"],keep=False)]
# x[x.duplicated(keep=False)]

,category,componentId,componentName,tagNumber,discipline,status,projectNumber
276023,CABLE,20820473,J-019GE0330,J-019GE0330,INSTRUMENTS,New,MOC983461
276247,CABLE,20820473,J-019GE0330,J-019GE0330,INSTRUMENTS,New,MOC983461
276024,CABLE,20820507,J-019GE0332,J-019GE0332,INSTRUMENTS,New,MOC983461
276248,CABLE,20820507,J-019GE0332,J-019GE0332,INSTRUMENTS,New,MOC983461
276025,CABLE,20820537,J-019GE0342,J-019GE0342,INSTRUMENTS,New,MOC983461
276249,CABLE,20820537,J-019GE0342,J-019GE0342,INSTRUMENTS,New,MOC983461
276022,CABLE,20813835,J-019XJB720,J-019XJB720,INSTRUMENTS,New,
276235,CABLE,20813835,J-019XJB720,J-019XJB720,INSTRUMENTS,New,
274221,CABLE,20196389,J-D19XJT623,J-D19XJT623,INSTRUMENTS,New,GOR350002
274870,CABLE,20196389,J-D19XJT623,J-D19XJT623,INSTRUMENTS,New,GOR350002


## Normalize discipline column Values

In [11]:
discipline_map_raw = {
    "INSTRUMENTS":"Instrumentation",
    "Instrumentation and Controls":"Instrumentation",
    "Instrumentation & Control":"Instrumentation",
    "Instrument & Controls":"Instrumentation",
    "Instrumentation & Control":"Instrumentation",
    "Instrumentation and Control":"Instrumentation",
    "InstrumentationandControls":"Instrumentation",
    "INSTRUMENT":"Instrumentation",
    
    "Instrument & Controls":"Instrumentation",
    
    "Instrument and Controls":"Instrumentation",
    "Instrumentation":"Instrumentation",


    "Telecommunications":"Telecoms",
    "Telecomunication":"Telecoms",
    "TELECOMS":"Telecoms",
    "Telecoms":"Telecoms",

    "Fire and Gas Safety":"FireAndGas",
    "FIRE_AND_GAS":"FireAndGas",
    "Safety":"FireAndGas",
    "FireAndGas":"FireAndGas",
    "Fire and Gas":"FireAndGas",
    

    "STRUCTURE":"Structural",
    "Structural":"Structural",
    "Subsea":"Subsea",

    "General General Multi Discipline":"General",
    "Marine":"General",

    # FIXED: garbage mapping direction
    "115SN2580": "Other",
}

# normalize dict keys to UPPER once
discipline_map = {str(k).strip().upper(): v for k, v in discipline_map_raw.items()}



def normalize_discipline(x):
    # keep blanks as blank
    if x is None or pd.isna(x):
        return None

    s = str(x).strip()
    if s == "":
        return None

    # optional: fix html encoding if your data has it
    s = s.replace("&amp;amp;", "&amp;").replace("&amp;", "&")

    # match using upper (so mapping keys must be uppercase)
    key = s.upper()

    # ✅ if mapped -> canonical, else keep original
    return discipline_map.get(key, s)

#    NOTE: TSV stores text; we serialize datetimes in ISO 8601 with timezone if present.
df_spi["discipline"] = df_spi["discipline"].map(normalize_discipline)

In [12]:
df_spi['discipline'].unique()

array(['Instrumentation', 'FireAndGas', 'Telecoms', None], dtype=object)

### Clean projectNumber column  and extract MocId

In [9]:
import pandas as pd
import re

def extract_mocs(value):
    """
    Extract and clean MOC numbers from the Project_no field.
    Returns a list of clean MOC values like 'MOC564805'.
    """

    if pd.isna(value):
        return []

    text = str(value)

    # Normalize separators
    text = text.replace("&", ",")
    text = text.replace(";", ",")
    text = text.replace("/", ",")
    text = text.replace(" and ", ",")
    text = text.replace("AND", ",")
    
    # Split multiple entries
    parts = re.split(r"[,\n]+", text)

    cleaned = []

    for p in parts:
        p = p.strip()

        # Find patterns with numbers and "MOC"
        match = re.search(r"(?:T?PMOC|TMOC|PMOC|MOC)[\s\-_]*([A-Za-z0-9]+)", p, re.IGNORECASE)

        if match:
            num = match.group(1).upper()
            cleaned.append("MOC" + num)
        else:
            # Catch patterns like "Rev 9 - MOC382271"
            match2 = re.search(r"MOC[\s\-_]*([A-Za-z0-9]+)", p, re.IGNORECASE)
            if match2:
                num = match2.group(1).upper()
                cleaned.append("MOC" + num)

    return cleaned


def clean_file(df_in: pd.DataFrame) -> pd.DataFrame:

    # Work on a copy to avoid mutating the caller's dataframe
    df = df_in.copy()

    # Create cleanMOC list column
    df["cleanMOC_list"] = df["projectNumber"].apply(extract_mocs)

    # Explode to multiple rows if more than one MOC
    df = df.explode("cleanMOC_list", ignore_index=True)

    # Rename final column
    df = df.rename(columns={"cleanMOC_list": "cleanMOC"})

    return df




In [10]:
df_spi_clean=clean_file(df_spi)

### Load Moc node and Tag node from SPI

In [6]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# =========================
# Connection settings
# =========================




URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")




# =========================
# Constants
# =========================
#TSV_PATH = "spi_cleaned.tsv"  # <-- change if needed
SOR_NAME = "spi_gorgon"
NULL_TOKENS = {"", "-", "--", "---", "unset", "*"}

# =========================
# Data prep (clean in Python)
# =========================

def prepare_spi_rows(df_in: pd.DataFrame) -> pd.DataFrame:
#def prepare_spi_rows(tsv_path: str):
    #df = pd.read_csv(tsv_path, sep="\t", dtype=str).fillna("")
    df=df_in.copy()
    # Trim whitespace
    #df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)
    df = df.map(lambda v: v.strip() if isinstance(v, str) else v)

    # Convert null-like tokens to None
    #df = df.applymap(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)
    df = df.map(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)

    # Convert real NaN to None
    df = df.where(pd.notna(df), None)

    rows = []
    for r in df.to_dict(orient="records"):
        mocId = r.get("cleanMOC")
        tag_num = r.get("tagNumber")

        # Only process rows that have both MOC id and tag_number
        if not mocId or not tag_num:
            continue

        # Exclude matching keys and source_file so t += row won't overwrite it
        props = {
            k: v for k, v in r.items()
            if k not in ["cleanMOC", "tagNumber", "sourceFile"] and v is not None
        }
        props["cleanMOC"] = mocId
        props["tagNumber"] = tag_num

        rows.append(props)

    print(f"Prepared {len(rows)} rows for upload.")
    return rows

# =========================
# Cypher (no MOC creation; Tag MERGE; append source_file; relate if MOC exists)
# =========================
UPSERT_CYPHER = """

UNWIND $rows AS row


MATCH (m:Moc {mocId: row.cleanMOC})
SET m.sourceFile =
  CASE
    WHEN m.sourceFile IS NULL THEN ['spi']
    WHEN 'spi' IN m.sourceFile THEN m.sourceFile
    ELSE m.sourceFile + ['spi']
  END

// Clean tag number
WITH row, m,
CASE
  WHEN row.tagNumber IS NOT NULL
       AND row.tagNumber = row.tagNumber              // exclude NaN
       AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]
  THEN row.tagNumber
  ELSE NULL
END AS tagVal

// Create/match Tag ONLY when tagVal is valid
FOREACH (_ IN CASE WHEN tagVal IS NULL THEN [] ELSE [1] END |
  MERGE (t:Tag {tagNumber: tagVal})
  SET
    // only allowed properties on Tag
    t.discipline = coalesce(row.discipline, t.discipline),
    // normalize (note: backslash must be escaped as \\\\ in Python, \\ in Cypher)
    t.tagNorm   = replace(replace(replace(replace(toLower(tagVal), "-", ""), "\\\\", ""), " ", ""), "f", ""),
    t.tagTokens = [tok IN split(tagVal, "-") WHERE trim(tok) <> "" | toLower(trim(tok))],
    // append-only source_file (no overwrite, no duplicates)
    t.sourceFile =
      CASE
        WHEN t.sourceFile IS NULL THEN ['spi_gorgon']
        WHEN 'spi_gorgon' IN t.sourceFile THEN t.sourceFile
        ELSE t.sourceFile + ['spi_gorgon']
      END, 
      t.siteName="Gorgon"
      
  MERGE (m)-[:ASSOCIATED_WITH_TAG]->(t)
)
"""





# =========================
# Run
# =========================
if __name__ == "__main__":
    

    rows = prepare_spi_rows(df_spi_clean)

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        with driver.session(database=DATABASE) as session:
            session.run(UPSERT_CYPHER, rows=rows, sor=SOR_NAME).consume()
        print("✅ Ingestion completed.")
    finally:
        driver.close()

Prepared 277569 rows for upload.
✅ Ingestion completed.


### Load spiRecord node

In [7]:
import pandas as pd
from neo4j import GraphDatabase
import numpy as np
import pandas as pd
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "Mangalore@1234"
DATABASE = "moc1201"

#URI = os.getenv("NEO4J_URI")
#USERNAME = os.getenv("NEO4J_USERNAME")
#PASSWORD = os.getenv("NEO4J_PASSWORD")
#DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))


# ======================================================
# 1. Ensure INDEXES — CRITICAL FOR SPEED!!!
# ======================================================
def ensure_indexes():
    with driver.session(database=DATABASE) as session:
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (s:spiRecord) ON (s.id)
        """)
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
        """)
    print("Indexes ensured.\n")


# ======================================================
# 2. FAST AURA‑OPTIMIZED CYPHER
# ======================================================
BATCH_QUERY = """
UNWIND $batch AS row
CREATE (s:SpiRecord)
set s.siteName= "Gorgon"

WITH s, row,
     [k IN keys(row.props_create)
      WHERE row.props_create[k] IS NOT NULL
        AND row.props_create[k] = row.props_create[k]
        AND NOT toString(row.props_create[k]) IN ["", "-", "--", "---", "unset", "*"]] AS keepKeys

FOREACH (k IN keepKeys | SET s[k] = row.props_create[k])

WITH s, row
WHERE row.tagNumber IS NOT NULL
  AND row.tagNumber = row.tagNumber
  AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]

MATCH (t:Tag {tagNumber: row.tagNumber})
MERGE (t)-[:PRESENT_IN]->(s);
"""


# ======================================================
# 3. Batch Upload Function (Super Fast)
# ======================================================


def upload_spi_fast(df_spi: pd.DataFrame, batch_size=5000):
    SENTINELS = {"", "-", "--", "---", "unset", "*"}

    print("Reading DF...")
    #df = pd.read_csv(spi_file, sep="\t", dtype=str).fillna("")
    df = df_spi.copy().fillna("")
    # Add ID column if missing
    if "id" not in df.columns:
        df.insert(0, "id", df.index.astype(str))

    # Vectorized strip
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

    # Rename project_status -> status
   # df = df.rename(columns={"project_status": "status"})

    # Only these columns exist (plus id). Keep them explicitly.
    PROP_COLS = [
        "category",
        "componentId",
        "componentName",
        "tagNumber",
        "discipline",
        "status",
        "projectNumber",
    ]

    print("Preparing rows...")
    rows = []
    for _, r in df[["id"] + PROP_COLS].iterrows():
        props_create = {}
        props_update = {}

        for col in PROP_COLS:
            val = r.get(col, "")

            # Skip NaN
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            # Normalize and filter sentinels/blanks
            if isinstance(val, str):
                v = val.strip()
                if v.lower() in SENTINELS:
                    continue
                val = v

            props_create[col] = val
            props_update[col] = val

        # Clean tag_number specifically for relationship logic
        tag_val = r.get("tagNumber", "")
        if tag_val is None or (isinstance(tag_val, float) and np.isnan(tag_val)):
            tag_val = ""
        else:
            t = str(tag_val).strip()
            tag_val = "" if t.lower() in SENTINELS else t

        rows.append({
            "id": str(r["id"]),
            "tagNumber": tag_val,
            "props_create": props_create,
            "props_update": props_update
        })

    print(f"Prepared {len(rows)} rows for upload.\n")

    # Write batches
    with driver.session(database=DATABASE) as session:
        batch = []
        batch_num = 0

        for i, row in enumerate(rows, start=1):
            batch.append(row)

            if len(batch) >= batch_size:
                batch_num += 1
                summary = session.run(BATCH_QUERY, batch=batch).consume().counters
                print(
                    f"Batch {batch_num} | "
                    f"{i}/{len(rows)} rows | "
                    f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
                )
                batch.clear()

        # Final batch
        if batch:
            batch_num += 1
            summary = session.run(BATCH_QUERY, batch=batch).consume().counters
            print(
                f"Final Batch {batch_num} | "
                f"{len(rows)}/{len(rows)} rows | "
                f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
            )

    print("\n🚀 Upload completed FAST.\n")


# ======================================================
# RUN
# ======================================================
ensure_indexes()
upload_spi_fast(df_spi)
driver.close()


Indexes ensured.

Reading DF...
Preparing rows...
Prepared 277533 rows for upload.

Batch 1 | 5000/277533 rows | Nodes=5000, Rels=8
Batch 2 | 10000/277533 rows | Nodes=5000, Rels=4
Batch 3 | 15000/277533 rows | Nodes=5000, Rels=3
Batch 4 | 20000/277533 rows | Nodes=5000, Rels=10
Batch 5 | 25000/277533 rows | Nodes=5000, Rels=2
Batch 6 | 30000/277533 rows | Nodes=5000, Rels=0
Batch 7 | 35000/277533 rows | Nodes=5000, Rels=0
Batch 8 | 40000/277533 rows | Nodes=5000, Rels=0
Batch 9 | 45000/277533 rows | Nodes=5000, Rels=0
Batch 10 | 50000/277533 rows | Nodes=5000, Rels=0
Batch 11 | 55000/277533 rows | Nodes=5000, Rels=0
Batch 12 | 60000/277533 rows | Nodes=5000, Rels=0
Batch 13 | 65000/277533 rows | Nodes=5000, Rels=0
Batch 14 | 70000/277533 rows | Nodes=5000, Rels=1
Batch 15 | 75000/277533 rows | Nodes=5000, Rels=6
Batch 16 | 80000/277533 rows | Nodes=5000, Rels=0
Batch 17 | 85000/277533 rows | Nodes=5000, Rels=9
Batch 18 | 90000/277533 rows | Nodes=5000, Rels=30
Batch 19 | 95000/277533 

# SPI Wheatstone

### WSU

### Read Data

In [8]:
import pandas as pd
import adlfs
from dotenv import load_dotenv

def abfss_to_adlfs_path(abfss_path: str) -> str:
    """
    Convert:
      abfss://container@account.dfs.core.windows.net/some/path/
    to:
      container/some/path
    """
    if not abfss_path.startswith("abfss://"):
        raise ValueError(f"Expected abfss:// path, got: {abfss_path}")

    without_scheme = abfss_path.replace("abfss://", "", 1)
    container, rest = without_scheme.split("@", 1)

    marker = ".dfs.core.windows.net/"
    if marker not in rest:
        raise ValueError(f"Unexpected ADLS Gen2 URL (missing {marker}): {abfss_path}")

    path = rest.split(marker, 1)[1]
    return f"{container}/{path}".rstrip("/")


def read_latest_parquet_from_adls(folder_abfss: str, storage_options: dict, **read_parquet_kwargs) -> pd.DataFrame:
    fs = adlfs.AzureBlobFileSystem(**storage_options)

    folder_fs = abfss_to_adlfs_path(folder_abfss).rstrip("/")

    # list with metadata (so we can get last_modified)
    items = fs.ls(folder_fs, detail=True)
    pq_items = [x for x in items if x.get("name", "").lower().endswith(".parquet")]

    if not pq_items:
        raise FileNotFoundError(f"No Parquet files found under: {folder_abfss}")

    latest = max(pq_items, key=lambda x: x.get("last_modified"))
    latest_path = latest["name"]

    print("Latest parquet:", latest_path)
    print("Last modified:", latest.get("last_modified"))

    # IMPORTANT: file handle -> DO NOT pass storage_options to pandas
    with fs.open(latest_path, "rb") as fp:
        return pd.read_parquet(fp, engine="pyarrow", **read_parquet_kwargs)


def main():
    load_dotenv()

    ACCOUNT_NAME = "chevrondatalake"
    CONTAINER_NAME = "produced"
    DIRECTORY_PATH = "Surface/source_hexagonsmartcloud_app3060720/SmartInstrumentation/WheatstoneUpstream/View/CVX01SI_WHEATUP.CVX_MTL"

    # folder path where parquet files exist
    base_path = f"abfss://{CONTAINER_NAME}@{ACCOUNT_NAME}.dfs.core.windows.net/{DIRECTORY_PATH}/"

    storage_options = {
        "account_name": ACCOUNT_NAME,
        "anon": False,   # uses your Azure login / environment auth
    }

    # ✅ read ONLY the latest parquet file from folder
    df = read_latest_parquet_from_adls(base_path, storage_options)

    print("SPI:", df.shape)
    return df


# In notebook:
df = main()
df_spi_u=df

Latest parquet: produced/Surface/source_hexagonsmartcloud_app3060720/SmartInstrumentation/WheatstoneUpstream/View/CVX01SI_WHEATUP.CVX_MTL/part-00000-f977b368-b53f-4739-80d9-a88570a1965a.c000.snappy.parquet
Last modified: 2026-03-08 08:38:01+00:00
SPI: (35216, 12)


In [114]:
df_spi_u=df

### Columns to keep

In [9]:
columns_to_keep=['CAT', 'CMPNT_ID','CMPNT_NAME', 'TAG_NUMBER', 'DISCIPLINE','STATUS', 'PROJECT_NUMBER']
df_spi_u=df_spi_u[columns_to_keep]
rename_map={'CAT':'category', 'CMPNT_ID':'componentId','CMPNT_NAME':'componentName',  'TAG_NUMBER':'tagNumber', 'DISCIPLINE': 'discipline','STATUS':'status', 'PROJECT_NUMBER': 'projectNumber'}
df_spi_u=df_spi_u.rename(columns=rename_map)
#df_spi_u.to_csv('SPI_CVX_MTL.tsv', sep="\t", index=False)

###  Clean projectNumber column  and extract MocId

In [10]:
df_spi_wsu_clean=clean_file(df_spi_u)

### Load Moc node and Tag node from SPI

In [11]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# =========================
# Connection settings
# =========================

#URI = os.getenv("NEO4J_URI")
#USERNAME = os.getenv("NEO4J_USERNAME")
#PASSWORD = os.getenv("NEO4J_PASSWORD")
#DATABASE = os.getenv("NEO4J_DATABASE")
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "Mangalore@1234"
DATABASE = "moc1201"
SOR_NAME = "spi_wsu"
NULL_TOKENS = {"", "-", "--", "---", "unset", "*"}

# =========================
# Data prep (clean in Python)
# =========================

def prepare_spi_rows(df_in: pd.DataFrame) -> pd.DataFrame:
#def prepare_spi_rows(tsv_path: str):
    #df = pd.read_csv(tsv_path, sep="\t", dtype=str).fillna("")
    df=df_in.copy()
    # Trim whitespace
    df = df.map(lambda v: v.strip() if isinstance(v, str) else v)

    # Convert null-like tokens to None
    df = df.map(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)

    # Convert real NaN to None
    df = df.where(pd.notna(df), None)

    rows = []
    for r in df.to_dict(orient="records"):
        mocId = r.get("cleanMOC")
        tag_num = r.get("tagNumber")

        # Only process rows that have both MOC id and tag_number
        if not mocId or not tag_num:
            continue

        # Exclude matching keys and source_file so t += row won't overwrite it
        props = {
            k: v for k, v in r.items()
            if k not in ["cleanMOC", "tagNumber", "sourceFile"] and v is not None
        }
        props["cleanMOC"] = mocId
        props["tagNumber"] = tag_num

        rows.append(props)

    print(f"Prepared {len(rows)} rows for upload.")
    return rows

# =========================
# Cypher (no MOC creation; Tag MERGE; append source_file; relate if MOC exists)
# =========================
UPSERT_CYPHER = """

UNWIND $rows AS row


MATCH (m:Moc {mocId: row.cleanMOC})
SET m.sourceFile =
  CASE
    WHEN m.sourceFile IS NULL THEN ['spi']
    WHEN 'spi' IN m.sourceFile THEN m.sourceFile
    ELSE m.sourceFile + ['spi']
  END

// Clean tag number
WITH row, m,
CASE
  WHEN row.tagNumber IS NOT NULL
       AND row.tagNumber = row.tagNumber              // exclude NaN
       AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]
  THEN row.tagNumber
  ELSE NULL
END AS tagVal

// Create/match Tag ONLY when tagVal is valid
FOREACH (_ IN CASE WHEN tagVal IS NULL THEN [] ELSE [1] END |
  MERGE (t:Tag {tagNumber: tagVal})
  SET
    // only allowed properties on Tag
    t.discipline = coalesce(row.discipline, t.discipline),
    // normalize (note: backslash must be escaped as \\\\ in Python, \\ in Cypher)
    t.tagNorm   = replace(replace(replace(replace(toLower(tagVal), "-", ""), "\\\\", ""), " ", ""), "f", ""),
    t.tagTokens = [tok IN split(tagVal, "-") WHERE trim(tok) <> "" | toLower(trim(tok))],
    // append-only source_file (no overwrite, no duplicates)
    t.sourceFile =
      CASE
        WHEN t.sourceFile IS NULL THEN ['spi_wsu']
        WHEN 'spi_wsu' IN t.sourceFile THEN t.sourceFile
        ELSE t.sourceFile + ['spi_wsu']
      END, 
      t.siteName="WSU"
      
  MERGE (m)-[:ASSOCIATED_WITH_TAG]->(t)
)
"""





# =========================
# Run
# =========================
if __name__ == "__main__":
    

    rows = prepare_spi_rows(df_spi_wsu_clean)

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        with driver.session(database=DATABASE) as session:
            session.run(UPSERT_CYPHER, rows=rows, sor=SOR_NAME).consume()
        print("✅ Ingestion completed.")
    finally:
        driver.close()

Prepared 35224 rows for upload.
✅ Ingestion completed.


### Load spiRecord node 

In [12]:
import pandas as pd
from neo4j import GraphDatabase
import numpy as np




import pandas as pd
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "Mangalore@1234"
DATABASE = "moc1201"
#SOR_NAME = "spi_wsu"

#URI = os.getenv("NEO4J_URI")
#USERNAME = os.getenv("NEO4J_USERNAME")
#PASSWORD = os.getenv("NEO4J_PASSWORD")
#DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))


# ======================================================
# 1. Ensure INDEXES — CRITICAL FOR SPEED!!!
# ======================================================
def ensure_indexes():
    with driver.session(database=DATABASE) as session:
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (s:spiRecord) ON (s.id)
        """)
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
        """)
    print("Indexes ensured.\n")


# ======================================================
# 2. FAST AURA‑OPTIMIZED CYPHER
# ======================================================
BATCH_QUERY = """
UNWIND $batch AS row
CREATE (s:SpiRecord)
set s.siteName= "WSU"

WITH s, row,
     [k IN keys(row.props_create)
      WHERE row.props_create[k] IS NOT NULL
        AND row.props_create[k] = row.props_create[k]
        AND NOT toString(row.props_create[k]) IN ["", "-", "--", "---", "unset", "*"]] AS keepKeys

FOREACH (k IN keepKeys | SET s[k] = row.props_create[k])

WITH s, row
WHERE row.tagNumber IS NOT NULL
  AND row.tagNumber = row.tagNumber
  AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]

MATCH (t:Tag {tagNumber: row.tagNumber})
MERGE (t)-[:PRESENT_IN]->(s);
"""


# ======================================================
# 3. Batch Upload Function (Super Fast)
# ======================================================


def upload_spi_fast(df_spi: pd.DataFrame, batch_size=5000):
    SENTINELS = {"", "-", "--", "---", "unset", "*"}

    print("Reading DF...")
    #df = pd.read_csv(spi_file, sep="\t", dtype=str).fillna("")
    df = df_spi.copy().fillna("")
    # Add ID column if missing
    if "id" not in df.columns:
        df.insert(0, "id", df.index.astype(str))

    # Vectorized strip
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

    # Rename project_status -> status
   # df = df.rename(columns={"project_status": "status"})

    # Only these columns exist (plus id). Keep them explicitly.
    PROP_COLS = [
        "category",
        "componentId",
        "componentName",
        "tagNumber",
        "discipline",
        "status",
        "projectNumber",
    ]

    print("Preparing rows...")
    rows = []
    for _, r in df[["id"] + PROP_COLS].iterrows():
        props_create = {}
        props_update = {}

        for col in PROP_COLS:
            val = r.get(col, "")

            # Skip NaN
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            # Normalize and filter sentinels/blanks
            if isinstance(val, str):
                v = val.strip()
                if v.lower() in SENTINELS:
                    continue
                val = v

            props_create[col] = val
            props_update[col] = val

        # Clean tag_number specifically for relationship logic
        tag_val = r.get("tagNumber", "")
        if tag_val is None or (isinstance(tag_val, float) and np.isnan(tag_val)):
            tag_val = ""
        else:
            t = str(tag_val).strip()
            tag_val = "" if t.lower() in SENTINELS else t

        rows.append({
            "id": str(r["id"]),
            "tagNumber": tag_val,
            "props_create": props_create,
            "props_update": props_update
        })

    print(f"Prepared {len(rows)} rows for upload.\n")

    # Write batches
    with driver.session(database=DATABASE) as session:
        batch = []
        batch_num = 0

        for i, row in enumerate(rows, start=1):
            batch.append(row)

            if len(batch) >= batch_size:
                batch_num += 1
                summary = session.run(BATCH_QUERY, batch=batch).consume().counters
                print(
                    f"Batch {batch_num} | "
                    f"{i}/{len(rows)} rows | "
                    f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
                )
                batch.clear()

        # Final batch
        if batch:
            batch_num += 1
            summary = session.run(BATCH_QUERY, batch=batch).consume().counters
            print(
                f"Final Batch {batch_num} | "
                f"{len(rows)}/{len(rows)} rows | "
                f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
            )

    print("\n🚀 Upload completed FAST.\n")


# ======================================================
# RUN
# ======================================================
ensure_indexes()
upload_spi_fast(df_spi_u)
driver.close()

Indexes ensured.

Reading DF...
Preparing rows...
Prepared 35216 rows for upload.

Batch 1 | 5000/35216 rows | Nodes=5000, Rels=64
Batch 2 | 10000/35216 rows | Nodes=5000, Rels=25
Batch 3 | 15000/35216 rows | Nodes=5000, Rels=24
Batch 4 | 20000/35216 rows | Nodes=5000, Rels=84
Batch 5 | 25000/35216 rows | Nodes=5000, Rels=0
Batch 6 | 30000/35216 rows | Nodes=5000, Rels=11
Batch 7 | 35000/35216 rows | Nodes=5000, Rels=50
Final Batch 8 | 35216/35216 rows | Nodes=216, Rels=0

🚀 Upload completed FAST.



# WSD

### Read Data

In [2]:
import pandas as pd
import adlfs
from dotenv import load_dotenv

def abfss_to_adlfs_path(abfss_path: str) -> str:
    """
    Convert:
      abfss://container@account.dfs.core.windows.net/some/path/
    to:
      container/some/path
    """
    if not abfss_path.startswith("abfss://"):
        raise ValueError(f"Expected abfss:// path, got: {abfss_path}")

    without_scheme = abfss_path.replace("abfss://", "", 1)
    container, rest = without_scheme.split("@", 1)

    marker = ".dfs.core.windows.net/"
    if marker not in rest:
        raise ValueError(f"Unexpected ADLS Gen2 URL (missing {marker}): {abfss_path}")

    path = rest.split(marker, 1)[1]
    return f"{container}/{path}".rstrip("/")


def read_latest_parquet_from_adls(folder_abfss: str, storage_options: dict, **read_parquet_kwargs) -> pd.DataFrame:
    fs = adlfs.AzureBlobFileSystem(**storage_options)

    folder_fs = abfss_to_adlfs_path(folder_abfss).rstrip("/")

    # list with metadata (so we can get last_modified)
    items = fs.ls(folder_fs, detail=True)
    pq_items = [x for x in items if x.get("name", "").lower().endswith(".parquet")]

    if not pq_items:
        raise FileNotFoundError(f"No Parquet files found under: {folder_abfss}")

    latest = max(pq_items, key=lambda x: x.get("last_modified"))
    latest_path = latest["name"]

    print("Latest parquet:", latest_path)
    print("Last modified:", latest.get("last_modified"))

    # IMPORTANT: file handle -> DO NOT pass storage_options to pandas
    with fs.open(latest_path, "rb") as fp:
        return pd.read_parquet(fp, engine="pyarrow", **read_parquet_kwargs)


def main():
    load_dotenv()

    ACCOUNT_NAME = "chevrondatalake"
    CONTAINER_NAME = "produced"
    DIRECTORY_PATH = "Surface/source_hexagonsmartcloud_app3060720/SmartInstrumentation/WheatstoneDownstream/View/CVX01SI_WHEATDN.CVX_MTL"

    # folder path where parquet files exist
    base_path = f"abfss://{CONTAINER_NAME}@{ACCOUNT_NAME}.dfs.core.windows.net/{DIRECTORY_PATH}/"

    storage_options = {
        "account_name": ACCOUNT_NAME,
        "anon": False,   # uses your Azure login / environment auth
    }

    # ✅ read ONLY the latest parquet file from folder
    df = read_latest_parquet_from_adls(base_path, storage_options)

    print("SPI:", df.shape)
    return df


# In notebook:
df = main()
df_spi_wsd=df

Latest parquet: produced/Surface/source_hexagonsmartcloud_app3060720/SmartInstrumentation/WheatstoneDownstream/View/CVX01SI_WHEATDN.CVX_MTL/part-00000-6f534abf-a67e-434a-be2f-f0d133ccf7c6.c000.snappy.parquet
Last modified: 2026-03-08 04:26:31+00:00
SPI: (189816, 14)


### Columns to keep

In [4]:
#df_spi_wsd.columns
columns_to_keep=['CAT', 'CMPNT_ID','CMPNT_NAME', 'TAG_NUMBER', 'DISCIPLINE','STATUS', 'PROJECT_NUMBER']
df_spi_wsd=df_spi_wsd[columns_to_keep]
rename_map={'CAT':'category', 'CMPNT_ID':'componentId','CMPNT_NAME':'componentName',  'TAG_NUMBER':'tagNumber', 'DISCIPLINE': 'discipline','STATUS':'status', 'PROJECT_NUMBER': 'projectNumber'}
df_spi_wsd=df_spi_wsd.rename(columns=rename_map)

### Clean projectNumber column and extract MocId

In [10]:

df_spi_wsd_clean=clean_file(df_spi_wsd)

### Load Moc node and Tag node from SPI

In [11]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# =========================
# Connection settings
# =========================

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "Mangalore@1234"
DATABASE = "moc1201"

#URI = os.getenv("NEO4J_URI")
#USERNAME = os.getenv("NEO4J_USERNAME")
#PASSWORD = os.getenv("NEO4J_PASSWORD")
#DATABASE = os.getenv("NEO4J_DATABASE")

# =========================
# Constants
# =========================
#TSV_PATH = "spi_cleaned.tsv"  # <-- change if needed
SOR_NAME = "spi_wsd"
NULL_TOKENS = {"", "-", "--", "---", "unset", "*"}

# =========================
# Data prep (clean in Python)
# =========================

def prepare_spi_rows(df_in: pd.DataFrame) -> pd.DataFrame:
#def prepare_spi_rows(tsv_path: str):
    #df = pd.read_csv(tsv_path, sep="\t", dtype=str).fillna("")
    df=df_in.copy()
    # Trim whitespace
    #df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)
    df = df.map(lambda v: v.strip() if isinstance(v, str) else v)

    # Convert null-like tokens to None
    #df = df.applymap(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)
    df = df.map(lambda v: None if (isinstance(v, str) and v in NULL_TOKENS) else v)

    # Convert real NaN to None
    df = df.where(pd.notna(df), None)

    rows = []
    for r in df.to_dict(orient="records"):
        mocId = r.get("cleanMOC")
        tag_num = r.get("tagNumber")

        # Only process rows that have both MOC id and tag_number
        if not mocId or not tag_num:
            continue

        # Exclude matching keys and source_file so t += row won't overwrite it
        props = {
            k: v for k, v in r.items()
            if k not in ["cleanMOC", "tagNumber", "sourceFile"] and v is not None
        }
        props["cleanMOC"] = mocId
        props["tagNumber"] = tag_num

        rows.append(props)

    print(f"Prepared {len(rows)} rows for upload.")
    return rows

# =========================
# Cypher (no MOC creation; Tag MERGE; append source_file; relate if MOC exists)
# =========================
UPSERT_CYPHER = """

UNWIND $rows AS row


MATCH (m:Moc {mocId: row.cleanMOC})
SET m.sourceFile =
  CASE
    WHEN m.sourceFile IS NULL THEN ['spi']
    WHEN 'spi' IN m.sourceFile THEN m.sourceFile
    ELSE m.sourceFile + ['spi']
  END

// Clean tag number
WITH row, m,
CASE
  WHEN row.tagNumber IS NOT NULL
       AND row.tagNumber = row.tagNumber              // exclude NaN
       AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]
  THEN row.tagNumber
  ELSE NULL
END AS tagVal

// Create/match Tag ONLY when tagVal is valid
FOREACH (_ IN CASE WHEN tagVal IS NULL THEN [] ELSE [1] END |
  MERGE (t:Tag {tagNumber: tagVal})
  SET
    // only allowed properties on Tag
    t.discipline = coalesce(row.discipline, t.discipline),
    // normalize (note: backslash must be escaped as \\\\ in Python, \\ in Cypher)
    t.tagNorm   = replace(replace(replace(replace(toLower(tagVal), "-", ""), "\\\\", ""), " ", ""), "f", ""),
    t.tagTokens = [tok IN split(tagVal, "-") WHERE trim(tok) <> "" | toLower(trim(tok))],
    // append-only source_file (no overwrite, no duplicates)
    t.sourceFile =
      CASE
        WHEN t.sourceFile IS NULL THEN ['spi_wsd']
        WHEN 'spi_wsd' IN t.sourceFile THEN t.sourceFile
        ELSE t.sourceFile + ['spi_wsd']
      END, 
      t.siteName="WSD"
      
  MERGE (m)-[:ASSOCIATED_WITH_TAG]->(t)
)
"""





# =========================
# Run
# =========================
if __name__ == "__main__":
    

    rows = prepare_spi_rows(df_spi_wsd_clean)

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        with driver.session(database=DATABASE) as session:
            session.run(UPSERT_CYPHER, rows=rows, sor=SOR_NAME).consume()
        print("✅ Ingestion completed.")
    finally:
        driver.close()

Prepared 189854 rows for upload.
✅ Ingestion completed.


### Load spiRecord 

In [5]:
import pandas as pd
from neo4j import GraphDatabase
import numpy as np
import pandas as pd
from neo4j import GraphDatabase



URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))


# ======================================================
# 1. Ensure INDEXES — CRITICAL FOR SPEED!!!
# ======================================================
def ensure_indexes():
    with driver.session(database=DATABASE) as session:
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (s:spiRecord) ON (s.id)
        """)
        session.run("""
            CREATE INDEX IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
        """)
    print("Indexes ensured.\n")


# ======================================================
# 2. FAST AURA‑OPTIMIZED CYPHER
# ======================================================
BATCH_QUERY = """
UNWIND $batch AS row
CREATE (s:SpiRecord)
set s.siteName= "WSD"

WITH s, row,
     [k IN keys(row.props_create)
      WHERE row.props_create[k] IS NOT NULL
        AND row.props_create[k] = row.props_create[k]
        AND NOT toString(row.props_create[k]) IN ["", "-", "--", "---", "unset", "*"]] AS keepKeys

FOREACH (k IN keepKeys | SET s[k] = row.props_create[k])

WITH s, row
WHERE row.tagNumber IS NOT NULL
  AND row.tagNumber = row.tagNumber
  AND NOT toString(row.tagNumber) IN ["", "-", "--", "---", "unset", "*"]

MATCH (t:Tag {tagNumber: row.tagNumber})
MERGE (t)-[:PRESENT_IN]->(s);
"""


# ======================================================
# 3. Batch Upload Function (Super Fast)
# ======================================================


def upload_spi_fast(df_spi: pd.DataFrame, batch_size=5000):
    SENTINELS = {"", "-", "--", "---", "unset", "*"}

    print("Reading DF...")
    #df = pd.read_csv(spi_file, sep="\t", dtype=str).fillna("")
    df = df_spi.copy().fillna("")
    # Add ID column if missing
    if "id" not in df.columns:
        df.insert(0, "id", df.index.astype(str))

    # Vectorized strip
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

    # Rename project_status -> status
   # df = df.rename(columns={"project_status": "status"})

    # Only these columns exist (plus id). Keep them explicitly.
    PROP_COLS = [
        "category",
        "componentId",
        "componentName",
        "tagNumber",
        "discipline",
        "status",
        "projectNumber",
    ]

    print("Preparing rows...")
    rows = []
    for _, r in df[["id"] + PROP_COLS].iterrows():
        props_create = {}
        props_update = {}

        for col in PROP_COLS:
            val = r.get(col, "")

            # Skip NaN
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            # Normalize and filter sentinels/blanks
            if isinstance(val, str):
                v = val.strip()
                if v.lower() in SENTINELS:
                    continue
                val = v

            props_create[col] = val
            props_update[col] = val

        # Clean tag_number specifically for relationship logic
        tag_val = r.get("tagNumber", "")
        if tag_val is None or (isinstance(tag_val, float) and np.isnan(tag_val)):
            tag_val = ""
        else:
            t = str(tag_val).strip()
            tag_val = "" if t.lower() in SENTINELS else t

        rows.append({
            "id": str(r["id"]),
            "tagNumber": tag_val,
            "props_create": props_create,
            "props_update": props_update
        })

    print(f"Prepared {len(rows)} rows for upload.\n")

    # Write batches
    with driver.session(database=DATABASE) as session:
        batch = []
        batch_num = 0

        for i, row in enumerate(rows, start=1):
            batch.append(row)

            if len(batch) >= batch_size:
                batch_num += 1
                summary = session.run(BATCH_QUERY, batch=batch).consume().counters
                print(
                    f"Batch {batch_num} | "
                    f"{i}/{len(rows)} rows | "
                    f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
                )
                batch.clear()

        # Final batch
        if batch:
            batch_num += 1
            summary = session.run(BATCH_QUERY, batch=batch).consume().counters
            print(
                f"Final Batch {batch_num} | "
                f"{len(rows)}/{len(rows)} rows | "
                f"Nodes={summary.nodes_created}, Rels={summary.relationships_created}"
            )

    print("\n🚀 Upload completed FAST.\n")


# ======================================================
# RUN
# ======================================================
ensure_indexes()
upload_spi_fast(df_spi_wsd)
driver.close()


Indexes ensured.

Reading DF...
Preparing rows...
Prepared 189816 rows for upload.

Batch 1 | 5000/189816 rows | Nodes=5000, Rels=64
Batch 2 | 10000/189816 rows | Nodes=5000, Rels=45
Batch 3 | 15000/189816 rows | Nodes=5000, Rels=111
Batch 4 | 20000/189816 rows | Nodes=5000, Rels=19
Batch 5 | 25000/189816 rows | Nodes=5000, Rels=64
Batch 6 | 30000/189816 rows | Nodes=5000, Rels=28
Batch 7 | 35000/189816 rows | Nodes=5000, Rels=11
Batch 8 | 40000/189816 rows | Nodes=5000, Rels=3
Batch 9 | 45000/189816 rows | Nodes=5000, Rels=57
Batch 10 | 50000/189816 rows | Nodes=5000, Rels=24
Batch 11 | 55000/189816 rows | Nodes=5000, Rels=100
Batch 12 | 60000/189816 rows | Nodes=5000, Rels=65
Batch 13 | 65000/189816 rows | Nodes=5000, Rels=29
Batch 14 | 70000/189816 rows | Nodes=5000, Rels=11
Batch 15 | 75000/189816 rows | Nodes=5000, Rels=8
Batch 16 | 80000/189816 rows | Nodes=5000, Rels=13
Batch 17 | 85000/189816 rows | Nodes=5000, Rels=6
Batch 18 | 90000/189816 rows | Nodes=5000, Rels=0
Batch 19 |